# Le Japon a perdu un actif sur six · *Japan has lost one working-age adult in six*

Notebook compagnon du chapitre **14. Démographie et croissance : comment la population façonne l'économie** — [lire l'article](https://nmlab.io/ressources/demographie-et-croissance).
Companion notebook to chapter **14. Demography and Growth: How Population Shapes the Economy** — [read the article](https://nmlab.io/en/ressources/demography-and-growth).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_working_age() -> Series:
    """Population en âge de travailler au Japon (15-64 ans), en millions, en direct de FRED.
    Japan's working-age population (15-64), in millions, live from FRED (LFWA64TTJPM647S)."""
    persons = nm.load_fred("LFWA64TTJPM647S", start="1970")
    return persons / 1_000_000

pop = load_working_age()


import matplotlib.dates as mdates
import pandas as pd
from matplotlib.figure import Figure

MONTHS_FR = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet",
             "août", "septembre", "octobre", "novembre", "décembre"]

LABELS = {
    "fr": dict(
        title="Le Japon a perdu un actif sur six",
        sub="Population en âge de travailler au Japon, 1970-2026",
        ylab="population de 15 à 64 ans, millions",
        total="pic de la population\nTOTALE : 2008",
        gap="13 ans d'écart",
        today="aujourd'hui"),
    "en": dict(
        title="Japan has lost one working-age adult in six",
        sub="Japan's working-age population, 1970-2026",
        ylab="population aged 15 to 64, millions",
        total="peak of the TOTAL\npopulation: 2008",
        gap="13-year gap",
        today="today"),
}

def peak_text(peak_date, peak_val: float, lang: str) -> str:
    """Étiquette du pic de 1995 (valeur et mois lus sur la série)."""
    if lang == "fr":
        value = f"{peak_val:.1f}".replace(".", ",")
        return f"pic : {value} M\n{MONTHS_FR[peak_date.month - 1]} {peak_date.year}"
    return f"peak: {peak_val:.1f}M\n{peak_date:%B %Y}"

def latest_text(latest_val: float, lang: str) -> str:
    """Étiquette du dernier point (valeur du jour)."""
    if lang == "fr":
        return f"{latest_val:.1f} M".replace(".", ",") + "\n" + LABELS["fr"]["today"]
    return f"{latest_val:.1f}M\n{LABELS['en']['today']}"

def caption(peak_val: float, latest_val: float, lang: str) -> str:
    """Note interne dynamique : recul de la population active depuis son pic."""
    decline = (peak_val - latest_val) / peak_val * 100
    if lang == "fr":
        value = f"{decline:.1f}".replace(".", ",")
        return ("La population active culmine en 1995, TREIZE ANS avant la population totale, et a reculé de "
                f"{value} %\ndepuis. La « stagnation » japonaise est d'abord un fait démographique. Source : Banque du Japon, FRED.")
    return ("The working-age population peaks in 1995, THIRTEEN YEARS before the total population, and has fallen "
            f"{decline:.1f}%\nsince. Japan's « stagnation » is first a demographic fact. Source: Bank of Japan, FRED.")

def build_figure(pop: Series, lang: str) -> Figure:
    """Population active du Japon, avec le pic de 1995 et le dernier point."""
    text = LABELS[lang]
    peak_date, peak_val = pop.idxmax(), pop.max()
    latest_date, latest_val = pop.index[-1], pop.iloc[-1]

    fig = nm.figure(height_px=1064)
    ax = nm.axes(fig, left=0.088)
    ax.plot(pop.index, pop, color=nm.COLORS["blue"], linewidth=2.9)

    # Repère du pic de la population TOTALE (2008), treize ans après.
    total_x = pd.Timestamp("2008-01-01")
    peak_x = pd.Timestamp("1995-03-01")
    ax.axvline(total_x, color=nm.COLORS["muted"], linestyle=(0, (2, 4)), linewidth=1.8, alpha=0.9)
    ax.text(pd.Timestamp("2008-10-01"), 71.0, text["total"], fontsize=20,
            color=nm.COLORS["muted"], va="center", ha="left", linespacing=1.4)
    ax.annotate("", xy=(total_x, 71.5), xytext=(peak_x, 71.5),
                arrowprops=dict(arrowstyle="<->", color=nm.COLORS["text"], lw=1.8))
    ax.text(pd.Timestamp("2001-09-01"), 72.5, text["gap"], fontsize=22, fontweight="bold",
            color=nm.COLORS["text"], ha="center", va="bottom")

    # Pic de la population active (rose) et dernier point (ambre).
    ax.scatter([peak_date], [peak_val], s=130, color=nm.COLORS["rose"], zorder=5)
    ax.annotate(peak_text(peak_date, peak_val, lang), xy=(peak_date, peak_val),
                xytext=(pd.Timestamp("1979-01-01"), 86.6), fontsize=22, fontweight="bold",
                color=nm.COLORS["rose"], ha="left", va="center", linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["rose"], lw=1.8))
    ax.scatter([latest_date], [latest_val], s=130, color=nm.COLORS["amber"], zorder=5)
    ax.annotate(latest_text(latest_val, lang), xy=(latest_date, latest_val),
                xytext=(pd.Timestamp("2015-06-01"), 80.3), fontsize=22, fontweight="bold",
                color=nm.COLORS["amber"], ha="left", va="center", linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["amber"], lw=1.8))

    ax.set_ylim(69, 91)
    ax.set_yticks(range(70, 91, 5))
    ax.set_ylabel(text["ylab"])
    ax.set_xlim(pd.Timestamp("1969-06-01"), pd.Timestamp("2027-06-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, caption(peak_val, latest_val, lang))
    return fig

build_figure(pop, LANG)